# DSA 4060 – Personalized Movie Recommender

**Student Name:** Samantha Masaki Nyatichi  
**Student ID:** 670455  
**Assigned User ID:** 16

### Assigned User ID calculation
The last two digits of Student ID 670455 are 55. Using the required formula:

\[
(55 \bmod 40) + 1 = 15 + 1 = 16
\]

This notebook examines only User 16's rating history, identifies patterns in their ratings, and uses movie genres and cosine similarity to recommend five movies they have not rated.

## Imports and setup

The notebook uses pandas to inspect and combine the datasets. `CountVectorizer` converts genre labels into numerical features, and cosine similarity compares the genre profiles of movies.

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

ASSIGNED_USER_ID = (55 % 40) + 1
print("Assigned User ID:", ASSIGNED_USER_ID)

Assigned User ID: 16


## Task 1 – Load and inspect the data (3 marks)

Load both supplied CSV files. This cell checks for the files in the project root first, then checks a `data` folder if the files have been stored there.

In [2]:
from pathlib import Path

def load_csv(filename):
    root_path = Path(filename)
    data_path = Path("data") / filename

    if root_path.exists():
        return pd.read_csv(root_path)
    if data_path.exists():
        return pd.read_csv(data_path)

    raise FileNotFoundError(
        f"Could not find {filename}. Put it beside this notebook "
        "or inside a folder named 'data'."
    )

movies = load_csv("movies.csv")
ratings = load_csv("ratings.csv")

print("movies.csv – first five rows")
display(movies.head())

print("ratings.csv – first five rows")
display(ratings.head())

movies.csv – first five rows


,movie_id,title,genres
0,1,Inception,Sci-Fi|Thriller
1,2,Interstellar,Sci-Fi|Drama
2,3,The Matrix,Sci-Fi|Action
3,4,Arrival,Sci-Fi|Drama
4,5,Edge of Tomorrow,Sci-Fi|Action


ratings.csv – first five rows


,user_id,movie_id,rating
0,1,2,3.5
1,1,3,4.5
2,1,5,3.5
3,1,6,3.5
4,1,11,3.0


In [3]:
print(f"movies.csv: {movies.shape[0]} rows and {movies.shape[1]} columns")
print(f"ratings.csv: {ratings.shape[0]} rows and {ratings.shape[1]} columns")

movies.csv: 36 rows and 3 columns
ratings.csv: 440 rows and 3 columns


In [4]:
print("Missing values in movies.csv:")
display(movies.isnull().sum())

print("Missing values in ratings.csv:")
display(ratings.isnull().sum())

print("Total missing values in movies.csv:", movies.isnull().sum().sum())
print("Total missing values in ratings.csv:", ratings.isnull().sum().sum())

Missing values in movies.csv:


movie_id    0
title       0
genres      0
dtype: int64

Missing values in ratings.csv:


user_id     0
movie_id    0
rating      0
dtype: int64

Total missing values in movies.csv: 0
Total missing values in ratings.csv: 0


### Task 1 – Interpretation

Use the dimensions and missing-value results shown above to complete this summary after running the cells. State the actual row and column counts, and describe any columns with missing values. If every count is zero, explain that no missing values were detected in these two files.

## Task 2 – Profile the assigned user (5 marks)

The following cells filter the ratings to User 16, join them to the movie catalogue, and display each rated movie with its genre labels and rating.

In [5]:
required_movie_columns = {"movie_id", "title", "genres"}
required_rating_columns = {"user_id", "movie_id", "rating"}

if not required_movie_columns.issubset(movies.columns):
    raise ValueError(f"movies.csv must contain: {sorted(required_movie_columns)}")
if not required_rating_columns.issubset(ratings.columns):
    raise ValueError(f"ratings.csv must contain: {sorted(required_rating_columns)}")

user_ratings = ratings.loc[
    ratings["user_id"] == ASSIGNED_USER_ID
].copy()

user_profile = (
    user_ratings.merge(
        movies[["movie_id", "title", "genres"]],
        on="movie_id",
        how="left",
        validate="many_to_one"
    )
    [["movie_id", "title", "genres", "rating"]]
    .sort_values(["rating", "title"], ascending=[False, True])
    .reset_index(drop=True)
)

print(f"Number of movies rated by User {ASSIGNED_USER_ID}: {len(user_profile)}")
display(user_profile)

Number of movies rated by User 16: 10


,movie_id,title,genres,rating
0,4,Arrival,Sci-Fi|Drama,4.0
1,20,Hidden Figures,Drama|Biography,4.0
2,6,Blade Runner 2049,Sci-Fi|Drama,3.5
3,3,The Matrix,Sci-Fi|Action,3.5
4,18,The Shawshank Redemption,Drama,3.5
5,5,Edge of Tomorrow,Sci-Fi|Action,3.0
6,33,Ford v Ferrari,Sports|Drama,3.0
7,32,Jumanji: Welcome to the Jungle,Comedy|Adventure,2.0
8,29,Sherlock Holmes,Mystery|Action,1.5
9,31,The Hangover,Comedy,1.5


In [6]:
if user_profile.empty:
    raise ValueError(
        f"No ratings were found for User {ASSIGNED_USER_ID}. "
        "Check the dataset and assigned user ID."
    )

top_three = user_profile.nlargest(3, "rating")
print("Three highest-rated movies for User 16:")
display(top_three[["title", "genres", "rating"]])

Three highest-rated movies for User 16:


,title,genres,rating
0,Arrival,Sci-Fi|Drama,4.0
1,Hidden Figures,Drama|Biography,4.0
2,Blade Runner 2049,Sci-Fi|Drama,3.5


### Genre preference analysis

A movie with multiple genre labels contributes its rating to each of those genres. The summary below reports the average rating and number of rated movies associated with each genre. Interpret the averages alongside the number of examples, since a genre represented by only one movie gives limited evidence.

In [7]:
genre_data = user_profile.dropna(subset=["genres"]).copy()
genre_data["genre"] = genre_data["genres"].astype(str).str.split("|")
genre_data = genre_data.explode("genre")
genre_data["genre"] = genre_data["genre"].str.strip()

genre_data = genre_data[
    genre_data["genre"].ne("")
    & genre_data["genre"].str.lower().ne("(no genres listed)")
]

genre_scores = (
    genre_data.groupby("genre")["rating"]
    .agg(avg_rating="mean", movies_rated="count")
    .sort_values(["avg_rating", "movies_rated"], ascending=[False, False])
)

display(genre_scores.round(2))

,avg_rating,movies_rated
genre,,
Biography,4.00,1
Drama,3.60,5
Sci-Fi,3.50,4
Sports,3.00,1
Action,2.67,3
Adventure,2.00,1
Comedy,1.75,2
Mystery,1.50,1


### Task 2 – Interpretation

After running the genre summary, identify the genre or genres with strong rating evidence. Mention specific highly rated movies and their ratings, and compare those examples with lower-rated movies where relevant. Avoid treating a genre with only one rated movie as a definite preference.

## Task 3 – Build the recommendation logic (7 marks)

### Step 1: Identify positively rated movies

For this quiz, a movie counts as one the user likes when its rating is 4.0 or higher.

In [8]:
liked = user_profile.loc[user_profile["rating"] >= 4.0].copy()

print(f"Movies rated 4.0 or higher by User {ASSIGNED_USER_ID}: {len(liked)}")
display(liked[["title", "genres", "rating"]])

if liked.empty:
    raise ValueError(
        "User 16 has no ratings of 4.0 or higher, so this positive-example "
        "recommendation method cannot proceed as specified."
    )

Movies rated 4.0 or higher by User 16: 2


,title,genres,rating
0,Arrival,Sci-Fi|Drama,4.0
1,Hidden Figures,Drama|Biography,4.0


### Step 2: Convert genres into numerical features

The genre strings use `|` as a separator. The custom tokenizer preserves each genre as one feature, rather than splitting a label such as `Sci-Fi` into separate words.

In [9]:
def split_genres(value):
    if pd.isna(value):
        return []
    return [
        genre.strip()
        for genre in str(value).split("|")
        if genre.strip()
        and genre.strip().lower() != "(no genres listed)"
    ]

movie_features = movies[["movie_id", "title", "genres"]].copy()
movie_features["genres"] = movie_features["genres"].fillna("")

vectorizer = CountVectorizer(
    tokenizer=split_genres,
    token_pattern=None,
    lowercase=False,
    binary=True
)

genre_matrix = vectorizer.fit_transform(movie_features["genres"])

print("Genre features:", list(vectorizer.get_feature_names_out()))
print("Feature matrix shape (movies × genres):", genre_matrix.shape)

Genre features: ['Action', 'Adventure', 'Animation', 'Biography', 'Comedy', 'Crime', 'Drama', 'Family', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Sports', 'Superhero', 'Thriller']
Feature matrix shape (movies × genres): (36, 16)


### Step 3: Calculate cosine similarity

Cosine similarity compares each movie's genre vector with the genre vectors of movies User 16 rated positively. A larger score means a closer match in genre representation, not a guarantee that the user will like the movie.

In [10]:
similarity_matrix = cosine_similarity(genre_matrix)

movie_index = {
    movie_id: position
    for position, movie_id in enumerate(movie_features["movie_id"])
}

liked_ids = liked.sort_values(
    ["rating", "title"], ascending=[False, True]
)["movie_id"].tolist()

liked_positions = [
    movie_index[movie_id]
    for movie_id in liked_ids
    if movie_id in movie_index
]

if not liked_positions:
    raise ValueError("The positively rated movies were not found in movies.csv.")

# Average similarity to the user's positively rated movies
scores = similarity_matrix[:, liked_positions].mean(axis=1)

# Strongest single match, used to explain each recommendation
best_match_positions = similarity_matrix[:, liked_positions].argmax(axis=1)
closest_liked_positions = [
    liked_positions[position] for position in best_match_positions
]
closest_similarity = similarity_matrix[:, liked_positions].max(axis=1)

movies_scored = movie_features.copy()
movies_scored["score"] = scores
movies_scored["closest_liked_position"] = closest_liked_positions
movies_scored["closest_similarity"] = closest_similarity

display(movies_scored.head())

,movie_id,title,genres,score,closest_liked_position,closest_similarity
0,1,Inception,Sci-Fi|Thriller,0.25,3,0.5
1,2,Interstellar,Sci-Fi|Drama,0.75,3,1.0
2,3,The Matrix,Sci-Fi|Action,0.25,3,0.5
3,4,Arrival,Sci-Fi|Drama,0.75,3,1.0
4,5,Edge of Tomorrow,Sci-Fi|Action,0.25,3,0.5


### Step 4: Exclude rated movies and select five candidates

Every movie already rated by User 16 is removed, regardless of whether its rating was high or low. The remaining movies are ranked by their average similarity to the user's positively rated movies. A stable title-based tie-breaker makes the result easier to reproduce.

In [11]:
rated_ids = set(user_ratings["movie_id"])

candidates = movies_scored.loc[
    ~movies_scored["movie_id"].isin(rated_ids)
].copy()

top5 = (
    candidates.sort_values(
        ["score", "closest_similarity", "title"],
        ascending=[False, False, True]
    )
    .head(5)
    .reset_index(drop=True)
)

print("Unrated movies available:", len(candidates))
print("Top five recommended titles:")
display(top5[["title", "genres", "score", "closest_similarity"]].round(3))

assert len(top5) == 5, (
    f"Expected five recommendations, but found {len(top5)}. "
    "Check that the catalogue contains at least five unrated movies."
)
assert top5["movie_id"].nunique() == 5
assert not set(top5["movie_id"]).intersection(rated_ids)

print(f"Validation passed: five unique recommendations; none was already rated by User {ASSIGNED_USER_ID}.")

Unrated movies available: 26
Top five recommended titles:


,title,genres,score,closest_similarity
0,Interstellar,Sci-Fi|Drama,0.75,1.0
1,The Pursuit of Happyness,Drama|Biography,0.75,1.0
2,The Social Network,Drama|Biography,0.75,1.0
3,Creed,Sports|Drama,0.50,0.5
4,Moneyball,Sports|Drama,0.50,0.5


Validation passed: five unique recommendations; none was already rated by User 16.


## Task 4 – Explain the recommendations (3 marks)

The results table pairs each recommended movie with its genres and an explanation grounded in the closest positively rated movie. The similarity score shown in the reason is the cosine similarity to that closest match.

In [12]:
liked_rating_lookup = liked.set_index("movie_id")["rating"].to_dict()

def recommendation_reason(row):
    closest_position = int(row["closest_liked_position"])
    closest_movie = movie_features.iloc[closest_position]

    recommended_genres = set(split_genres(row["genres"]))
    liked_genres = set(split_genres(closest_movie["genres"]))
    shared_genres = sorted(recommended_genres.intersection(liked_genres))

    liked_rating = liked_rating_lookup.get(closest_movie["movie_id"])
    similarity = row["closest_similarity"]

    if shared_genres:
        genre_text = ", ".join(shared_genres)
        return (
            f"Shares {genre_text} with '{closest_movie['title']}', "
            f"which User 16 rated {liked_rating:.1f}. "
            f"Cosine similarity to that movie: {similarity:.2f}."
        )

    return (
        f"Ranked by its genre similarity to '{closest_movie['title']}', "
        f"which User 16 rated {liked_rating:.1f}. "
        f"No genre labels overlap; cosine similarity: {similarity:.2f}."
    )

results = pd.DataFrame({
    "Rank": range(1, len(top5) + 1),
    "Recommended Movie": top5["title"],
    "Genres": top5["genres"],
    "Reason": [recommendation_reason(row) for _, row in top5.iterrows()]
})

pd.set_option("display.max_colwidth", None)
display(results)

,Rank,Recommended Movie,Genres,Reason
0,1,Interstellar,Sci-Fi|Drama,"Shares Drama, Sci-Fi with 'Arrival', which User 16 rated 4.0. Cosine similarity to that movie: 1.00."
1,2,The Pursuit of Happyness,Drama|Biography,"Shares Biography, Drama with 'Hidden Figures', which User 16 rated 4.0. Cosine similarity to that movie: 1.00."
2,3,The Social Network,Drama|Biography,"Shares Biography, Drama with 'Hidden Figures', which User 16 rated 4.0. Cosine similarity to that movie: 1.00."
3,4,Creed,Sports|Drama,"Shares Drama with 'Arrival', which User 16 rated 4.0. Cosine similarity to that movie: 0.50."
4,5,Moneyball,Sports|Drama,"Shares Drama with 'Arrival', which User 16 rated 4.0. Cosine similarity to that movie: 0.50."


## Task 5 – Critical thinking (2 marks)

### Limitation

This recommender uses only genre labels, so it cannot distinguish between movies with similar genres but very different stories or quality. The catalogue is also limited, which may restrict the number of strong options available after previously rated movies are excluded.

### Suggested improvement

A useful next step would be a hybrid recommender that combines genre similarity with collaborative filtering. Collaborative filtering could use rating patterns from users with similar tastes, while genre information would continue to connect recommendations to User 16's observed interests.